# Your First Machine Learning Project in Python

**Hey friends!** You are building a tiny flower classifier. Your mission is to beat a simple baseline, choose a model using validation data, and explain a mistake.

Use Google Colab → File → Upload notebook. Run cells in order. Python basics help; no GPU or external data download is needed.

Iris is a small educational dataset with 150 examples, four measurements and three species. It makes the workflow easy to inspect. Its results do not establish performance on new real-world populations.

Write a prediction before each experiment. Answers and explanations are at the end.

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__)
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
print(X.head().to_string(index=False))
print("Species:", dict(enumerate(iris.target_names)))
print("Examples:", len(X), "| Missing values:", int(X.isna().sum().sum()))

## 1. Predict before splitting

Which columns are inputs? Which value are we trying to predict? Why would including the true species among the input columns spoil the experiment?

We reserve 30 examples for a final test. The remaining examples become 90 training and 30 validation rows. Stratification preserves class proportions. This random split suits this educational dataset; time-dependent or grouped data needs a suitable split.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42
)
print("Train / validation / test:", len(X_train), len(X_valid), len(X_test))
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_score = accuracy_score(y_valid, baseline.predict(X_valid))
print("Validation baseline:", round(baseline_score, 3))

## 2. Make your first prediction

Our model compares a flower with its nearest neighbours. Feature scaling gives measurements a comparable scale. The pipeline learns scaling from training data.

**Before running:** predict whether this model will beat the baseline. What would a tie tell you?

In [ ]:
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
model.fit(X_train, y_train)
valid_predictions = model.predict(X_valid)
print("Validation accuracy:", round(accuracy_score(y_valid, valid_predictions), 3))
print("Correct predictions:", int((valid_predictions == y_valid).sum()), "/", len(y_valid))

## Your next learning steps

[Get the Free 90-Day ML Roadmap PDF](https://how-to-learn-machine-learning.kit.com/7a25f667cb) for a structured learning plan and weekly tips by email.

## 3. Find a combination and defend it

Try k = 1, 3, 5, 9 and 15 with two inputs (petal measurements) or all four. You have ten combinations. Predict which will win and why.

Fill in your reasoning before running the experiment. The checks ask for a written hypothesis. They cannot judge its quality.

In [ ]:
hypothesis = ""  # Write your prediction and reasoning here.
ready = len(hypothesis.strip()) >= 15
print("Ready to experiment." if ready else "Write at least one sentence in hypothesis, then rerun this cell.")
feature_sets = {
    "petals": ["petal length (cm)", "petal width (cm)"],
    "all": list(X.columns),
}
results = []
if ready:
    for feature_name, columns in feature_sets.items():
        for k in [1, 3, 5, 9, 15]:
            candidate = make_pipeline(
                StandardScaler(), KNeighborsClassifier(n_neighbors=k)
            )
            candidate.fit(X_train[columns], y_train)
            score = accuracy_score(y_valid, candidate.predict(X_valid[columns]))
            results.append({"features": feature_name, "k": k, "validation_accuracy": score})
    print(pd.DataFrame(results).sort_values("validation_accuracy", ascending=False).to_string(index=False))

## 4. Lock your choice, then evaluate once

Choose a combination using validation results. In a tie, decide before opening the test results. Record your choice and explanation below. Refit on the combined training and validation data, then evaluate on the reserved test set.

The final cell runs only after you write a reason and turn the switch on. It prevents accidental repeat evaluation within the same runtime. Restarting a runtime resets the guard, so responsible test use still depends on you.

In [ ]:
chosen_features = "all"  # Choose "all" or "petals".
chosen_k = 5
choice_reason = ""  # Explain your choice using validation results.
run_final_test = False  # Set True only when your decision is final.

if not run_final_test or len(choice_reason.strip()) < 15 or not ready:
    print("Complete the experiment, explain your choice, then enable the final test.")
elif globals().get("final_test_done", False):
    print("The test was already evaluated. Report it; use validation data for further experiments.")
else:
    assert chosen_features in feature_sets, "Choose all or petals."
    assert chosen_k in [1, 3, 5, 9, 15], "Choose a tested k."
    columns = feature_sets[chosen_features]
    final_model = make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=chosen_k)
    )
    final_model.fit(X_dev[columns], y_dev)
    test_predictions = final_model.predict(X_test[columns])
    final_baseline = DummyClassifier(strategy="most_frequent").fit(X_dev, y_dev)
    print("Test baseline:", round(accuracy_score(y_test, final_baseline.predict(X_test)), 3))
    print("Test model:", round(accuracy_score(y_test, test_predictions), 3))
    print("Confusion matrix: rows = actual, columns = predicted; species order:", iris.target_names)
    print(confusion_matrix(y_test, test_predictions, labels=[0, 1, 2]))
    mistakes = X_test.copy()
    mistakes["actual"] = iris.target_names[y_test.to_numpy()]
    mistakes["predicted"] = iris.target_names[test_predictions]
    print(mistakes.loc[y_test.to_numpy() != test_predictions].to_string(index=False))
    final_test_done = True

## 5. Explain what you built

Write a short project README with your question, data source, split, baseline, validation choice, final test score and limitations. If there are no test errors, explain why 30 correct predictions would still leave uncertainty about future performance.

[Use our portfolio README template](https://howtolearnmachinelearning.com/articles/machine-learning-portfolio/).

## Answers and explanations

1. The four measurement columns are inputs. Species is the target. Including the target as an input leaks the answer.
2. Each split is balanced. Always predicting one species gives a baseline accuracy of about 0.333. A model should demonstrate improvement on the same evaluation data.
3. Training fits model parameters. Validation informs your choices. The test set estimates performance after those choices are fixed.
4. Several combinations can tie. There is no universal winning k or feature set. The supplied answer experiment shows the ranking for this particular split. A difference of one validation case changes accuracy by about 3.3 percentage points.
5. Fitting the scaler within the pipeline prevents validation or test statistics from influencing training preprocessing.
6. Refit your selected pipeline on development data before the final test. The test score can be lower than validation and should be reported honestly.
7. A small, clean educational dataset offers limited evidence about real-world generalisation. Repeatedly tuning to this test set would compromise its role.

### Worked validation answer

Run this after your own attempt. It uses training and validation data only.

In [ ]:
answer_rows = []
for feature_name, columns in feature_sets.items():
    for k in [1, 3, 5, 9, 15]:
        answer_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
        answer_model.fit(X_train[columns], y_train)
        answer_rows.append({
            "features": feature_name,
            "k": k,
            "validation_accuracy": accuracy_score(y_valid, answer_model.predict(X_valid[columns]))
        })
print(pd.DataFrame(answer_rows).sort_values(
    ["validation_accuracy", "k", "features"], ascending=[False, True, True]
).to_string(index=False))

## Keep building

[Explore the ML Starter Kit](https://howtolearnmachinelearning.com/starter-kit/) for a guided companion to your learning. [Continue with model evaluation](https://howtolearnmachinelearning.com/articles/evaluating-machine-learning-models/).

Sources: [scikit-learn Iris dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html), [data leakage and pipelines](https://scikit-learn.org/stable/common_pitfalls.html). Data supplied with scikit-learn; see the dataset documentation for provenance.

Created for How to Learn Machine Learning. 